# Walking Joint Kinematics Animation

Creates animations of joint angles during walking using Ray parallel rendering.

Layout:
- Row 0-1: Coxa flexion angles (all 6 legs)
- Row 2-3: Femur angles (all 6 legs)
- Row 4-5: Tibia angles (all 6 legs)
- Row 6+: MuJoCo render / Video

## 1. Configuration

In [ ]:
# =============================================================================
# USER CONFIGURATION
# =============================================================================

# Data paths
H5_PATH = '/home/user/src/JARVIS-HybridNet/projects/fly50_V4/predictions/predictions3D/Predictions_3D_20260121-104455/Fruitfly_ik_V1_free.h5'
MUJOCO_MODEL_PATH = '/home/user/src/Fly_tracking/assets/fruitfly_v1/fruitfly_v1_free.xml'

# Video sync
VIDEO_FRAME_START = 0  # Frame in video corresponding to frame 0 of H5
CAMERA_ID = 'Cam2012630'

# Frame rates
DATA_FPS = 800
OUTPUT_FPS = 30

# Output
OUTPUT_DIR = './output/walking_animation'

print("Configuration loaded.")

## 2. Imports and Setup

In [ ]:
import numpy as np
import h5py
import yaml
import mujoco
from pathlib import Path
from tqdm.auto import tqdm
import cv2
import ray
import time as time_module
import warnings
warnings.filterwarnings('ignore')

OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Initialize Ray
if not ray.is_initialized():
    ray.init()
print(f"Ray initialized with {ray.cluster_resources().get('CPU', 0)} CPUs")

## 3. Load Data

In [ ]:
# Load IK data
with h5py.File(H5_PATH, 'r') as f:
    qpos = f['qpos'][:]
    names_qpos = [n.decode() for n in f['names_qpos'][:]]

n_frames = len(qpos)
time = np.arange(n_frames) / DATA_FPS

print(f"Loaded IK data: {n_frames} frames ({time[-1]:.3f}s)")
print(f"qpos shape: {qpos.shape}")

In [ ]:
# Get video path from info.yaml
info_yaml = Path(H5_PATH).parent / 'info.yaml'
VIDEO_PATH = None

if info_yaml.exists():
    with open(info_yaml) as f:
        info = yaml.safe_load(f)
    recording_path = Path(info['recording_path'])
    VIDEO_PATH = recording_path / f"{CAMERA_ID}.mp4"
    print(f"Video path: {VIDEO_PATH}")
    print(f"Exists: {VIDEO_PATH.exists()}")

In [ ]:
# Load video frames
video_frames = None

if VIDEO_PATH and VIDEO_PATH.exists():
    print(f"Loading {n_frames} video frames starting at {VIDEO_FRAME_START}...")
    cap = cv2.VideoCapture(str(VIDEO_PATH))
    cap.set(cv2.CAP_PROP_POS_FRAMES, VIDEO_FRAME_START)
    
    video_frames = []
    for _ in tqdm(range(n_frames), desc="Loading video"):
        ret, frame = cap.read()
        if not ret:
            break
        video_frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    video_frames = np.array(video_frames)
    print(f"Loaded video: {video_frames.shape}")
else:
    print("No video - will use MuJoCo render only")

In [ ]:
# Build joint indices for the joints we want to plot
LEGS = ['T1_left', 'T1_right', 'T2_left', 'T2_right', 'T3_left', 'T3_right']
JOINTS = ['coxa_flexion', 'femur', 'tibia']

joint_indices = {}
for leg in LEGS:
    for joint in JOINTS:
        name = f"{joint}_{leg}"
        if name in names_qpos:
            joint_indices[name] = names_qpos.index(name)

print(f"Built indices for {len(joint_indices)} joints")

## 4. Render Function (Ray Remote)

In [ ]:
import matplotlib
matplotlib.use('agg')  # Use non-interactive backend for Ray workers

@ray.remote
def render_frame(frame_idx, vid_ref, qpos_ref, time_ref, config):
    """
    Render a single frame with video and joint angle time series plots.
    Uses Ray remote function with object references for efficient data sharing.
    
    Layout:
    - Row 0-1: Coxa flexion angles (all 6 legs)
    - Row 2-3: Femur angles (all 6 legs)
    - Row 4-5: Tibia angles (all 6 legs)
    - Row 6+: Video frame (full width)
    """
    import matplotlib.gridspec as gridspec
    import matplotlib.pyplot as plt
    import numpy as np
    
    # Extract config
    figsize = config.get('figsize', (15, 12))
    dpi = config.get('dpi', 100)
    trail = config.get('trail', 50)  # Number of frames to highlight
    joint_indices = config.get('joint_indices', {})
    legs = config.get('legs', ['T1_left', 'T1_right', 'T2_left', 'T2_right', 'T3_left', 'T3_right'])
    joints = config.get('joints', ['coxa_flexion', 'femur', 'tibia'])
    leg_labels = config.get('leg_labels', {'T1_left': 'L1', 'T1_right': 'R1', 'T2_left': 'L2', 'T2_right': 'R2', 'T3_left': 'L3', 'T3_right': 'R3'})
    joint_labels = config.get('joint_labels', {'coxa_flexion': 'Coxa Flex', 'femur': 'Femur', 'tibia': 'Tibia'})
    video_frame_start = config.get('video_frame_start', 0)
    
    # Leg colors
    leg_colors = ['C0', 'C1', 'C2', 'C3', 'C4', 'C5']
    
    # Create figure with GridSpec
    fig = plt.figure(figsize=figsize, dpi=dpi, constrained_layout=True)
    gs = gridspec.GridSpec(10, 1, figure=fig, hspace=0.01, wspace=0.03)
    
    # Current time
    current_time = time_ref[frame_idx]
    
    # Plot each joint type
    for joint_idx, joint in enumerate(joints):
        ax = fig.add_subplot(gs[joint_idx * 2:joint_idx * 2 + 2, 0])
        
        for leg_idx, leg in enumerate(legs):
            name = f"{joint}_{leg}"
            if name not in joint_indices:
                continue
            
            qpos_idx = joint_indices[name]
            angles = np.degrees(qpos_ref[:, qpos_idx])
            color = leg_colors[leg_idx]
            
            # Plot full time series with low alpha
            ax.plot(time_ref, angles, alpha=0.3, linewidth=1, color=color)
            
            # Highlight trail
            start_idx = max(0, frame_idx - trail)
            end_idx = frame_idx + 1
            ax.plot(time_ref[start_idx:end_idx], angles[start_idx:end_idx],
                   alpha=1.0, linewidth=2, color=color, label=leg_labels[leg])
        
        # Add vertical line at current time
        ax.axvline(current_time, color='r', linestyle='--', linewidth=1, alpha=0.7)
        
        ax.set_title(f'{joint_labels.get(joint, joint)} (deg)', fontsize=10)
        ax.set_xlabel('Time (s)', fontsize=8)
        ax.set_ylabel('Angle', fontsize=8)
        ax.tick_params(labelsize=7)
        
        if joint_idx == 0:
            ax.legend(frameon=False, loc='upper right', ncol=6, fontsize=8,
                     labelcolor='linecolor', handlelength=0, handleheight=0, columnspacing=0.5)
    
    # Video frame (bottom rows)
    ax_video = fig.add_subplot(gs[6:, :])
    if vid_ref is not None:
        ax_video.imshow(vid_ref[frame_idx])
    else:
        ax_video.text(0.5, 0.5, 'No video', ha='center', va='center', fontsize=20)
    ax_video.axis('off')
    ax_video.set_title(f'Frame {video_frame_start + frame_idx} (t={current_time:.3f}s)', fontsize=10)

    # Convert figure to numpy array
    fig.canvas.draw()
    width, height = int(fig.get_figwidth() * fig.dpi), int(fig.get_figheight() * fig.dpi)
    buf = fig.canvas.buffer_rgba()
    image = np.asarray(buf).reshape(height, width, 4)[:, :, :3]  # Remove alpha channel
    
    plt.close(fig)
    
    return image

print("Ray render function defined.")

In [ ]:
def render_animation_ray(vid, qpos, time, config=None, frames_to_render=None):
    """
    Render animation frames in parallel using Ray.
    """
    if config is None:
        config = {}
    
    if frames_to_render is None:
        frames_to_render = list(range(len(qpos)))
    
    print(f"Rendering {len(frames_to_render)} frames using Ray...")
    if vid is not None:
        print(f"Video shape: {vid.shape}, qpos shape: {qpos.shape}")
    else:
        print(f"No video, qpos shape: {qpos.shape}")
    
    start_time = time_module.time()
    
    # Put large arrays into Ray's shared object store
    vid_ref = ray.put(vid) if vid is not None else None
    qpos_ref = ray.put(qpos)
    time_ref = ray.put(time)
    
    # Launch Ray tasks for all frames
    result_refs = []
    for frame_idx in frames_to_render:
        result_refs.append(render_frame.remote(frame_idx, vid_ref, qpos_ref, time_ref, config))
    
    # Get results with progress bar
    results = []
    for ref in tqdm(result_refs, desc="Rendering"):
        results.append(ray.get(ref))
    
    elapsed = time_module.time() - start_time
    print(f"Rendered {len(results)} frames in {elapsed:.1f}s ({len(results)/elapsed:.1f} fps)")
    
    return np.array(results)

print("Ray animation renderer defined.")

## 5. Test Single Frame

In [ ]:
# Config for rendering
config = {
    'figsize': (15, 12),
    'dpi': 100,
    'trail': 50,
    'joint_indices': joint_indices,
    'legs': LEGS,
    'joints': JOINTS,
    'leg_labels': {'T1_left': 'L1', 'T1_right': 'R1', 'T2_left': 'L2', 'T2_right': 'R2', 'T3_left': 'L3', 'T3_right': 'R3'},
    'joint_labels': {'coxa_flexion': 'Coxa Flex', 'femur': 'Femur', 'tibia': 'Tibia'},
    'video_frame_start': VIDEO_FRAME_START
}

# Test single frame (without Ray for quick test)
test_idx = n_frames // 2
test_img = ray.get(render_frame.remote(test_idx, 
                                        ray.put(video_frames) if video_frames is not None else None,
                                        ray.put(qpos), 
                                        ray.put(time), 
                                        config))

import matplotlib.pyplot as plt
plt.figure(figsize=(15, 12))
plt.imshow(test_img)
plt.axis('off')
plt.savefig(OUTPUT_DIR / 'test_frame.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Test frame shape: {test_img.shape}")

## 6. Render Full Animation

In [ ]:
# Select frames to render (e.g., every 4th frame for speed)
FRAME_STEP = 4
frames_to_render = list(range(0, n_frames, FRAME_STEP))
print(f"Will render {len(frames_to_render)} frames (every {FRAME_STEP}th)")
print(f"Output duration: {len(frames_to_render) / OUTPUT_FPS:.1f}s at {OUTPUT_FPS} fps")

In [ ]:
# Render all frames in parallel
rendered_frames = render_animation_ray(
    vid=video_frames,
    qpos=qpos,
    time=time,
    config=config,
    frames_to_render=frames_to_render
)
print(f"Rendered frames shape: {rendered_frames.shape}")

In [ ]:
# Save as video
output_path = OUTPUT_DIR / 'walking_animation.mp4'
h, w = rendered_frames[0].shape[:2]
out = cv2.VideoWriter(str(output_path), cv2.VideoWriter_fourcc(*'mp4v'), OUTPUT_FPS, (w, h))
for frame in tqdm(rendered_frames, desc="Writing video"):
    out.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
out.release()
print(f"Saved: {output_path}")

## 7. Display Video

In [ ]:
# Display video inline (if mediapy works) or show file path
try:
    import mediapy as media
    media.show_video(rendered_frames, fps=OUTPUT_FPS)
except ImportError:
    print(f"Video saved to: {output_path}")
    print("Install mediapy to display inline: pip install mediapy")
except Exception as e:
    # mediapy can fail with codec issues on some systems
    print(f"Inline display failed ({type(e).__name__}), but video was saved successfully.")
    print(f"Video saved to: {output_path.resolve()}")